Code-only archive: embedded input/reference bundles were omitted. Run the adjacent build_notebook.py or build_notebooks.py with your local inputs to recreate the executable Kaggle notebook.


# PROTEUS: single Lag4 vs end-to-end attention SPD

No forest. Same41channels,band4–40Hz,lag4 and10folds. Baseline: single OAS+causal tangent+LR. Alternatives: attention Gram+ridge and bounded score exponential. IndependentQ/K. Full gradient through prior,cumulative session reference,tangent/scaler and retained final LR. No discarded auxiliary head. Private T4 execution;no competition submission.

In [ ]:
import os,sys,pathlib,json,hashlib,zipfile,io,base64,subprocess
os.environ.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",PYTHONUNBUFFERED="1",MPLCONFIGDIR="/tmp/single-mpl")
import torch
assert torch.cuda.is_available(),"Enable requested Kaggle GPU"
print("GPU",torch.cuda.get_device_name(0),flush=True)
project=pathlib.Path("/kaggle/working/riemann-project");project.mkdir(exist_ok=True)
sources=list(pathlib.Path("/kaggle/input").rglob("source-integrity.json"));assert len(sources)==1
raw=sources[0].read_bytes()
assert hashlib.sha256(raw).hexdigest()=="3e8cf0b952a1a18ab496a89d6a31275522e8efa003ea6b8ed4b9e6467d2a3c81"
for rel,digest in json.loads(raw).items():
    p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
    data=(sources[0].parent/rel).read_bytes();assert hashlib.sha256(data).hexdigest()==digest
    dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
payload=base64.b64decode(__import__("pathlib").Path("local-bundle.base64").read_text())
assert hashlib.sha256(payload).hexdigest()=="d3a0835d1dff95fb278aa61073bc8602c5b2711da9d38ef6277adfe7b86e80f9"
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    manifest=json.loads(z.read("single-manifest.json"))
    for rel,digest in manifest.items():
        p=pathlib.PurePosixPath(rel);assert not p.is_absolute() and ".." not in p.parts
        data=z.read(rel);assert hashlib.sha256(data).hexdigest()==digest
        dest=project/rel;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(data)
repro=project/"reproduction";experiment=repro/"single_attention_ablation"
inputs=list(pathlib.Path("/kaggle/input").rglob("input-integrity.json"));assert len(inputs)==1
input_manifest=json.loads(inputs[0].read_text());input_dir=inputs[0].parent
for name,digest in input_manifest["files_sha256"].items():
    h=hashlib.sha256()
    with (input_dir/name).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    assert h.hexdigest()==digest,name
(repro/"data").mkdir(exist_ok=True)
for split in ["train","val","test"]:
    (repro/"data"/f"proteus_{split}.npz").symlink_to(input_dir/f"proteus_{split}.npz")
(repro/"temporal_spd_research/cache").symlink_to(input_dir,target_is_directory=True)
envdir=pathlib.Path("/tmp/single-env")
subprocess.run([sys.executable,"-m","venv","--without-pip","--system-site-packages",str(envdir)],check=True)
python=str(envdir/"bin/python")
pins=["numpy==2.5.4","scipy==1.18.1","scikit-learn==1.9.1","pyriemann==0.12","pandas==3.1.0","threadpoolctl==3.7.0"]
subprocess.run([sys.executable,"-m","pip","--python",python,"install","--quiet",*pins],check=True)
out=pathlib.Path("/kaggle/working/single-results");out.mkdir(exist_ok=True)
env=dict(os.environ,SINGLE_OUTPUT=str(out))
(out/"bundle-provenance.json").write_text(json.dumps(dict(single_bundle_sha256="d3a0835d1dff95fb278aa61073bc8602c5b2711da9d38ef6277adfe7b86e80f9",original_source_manifest_sha256=hashlib.sha256(raw).hexdigest(),input_manifest=input_manifest,source_manifest=manifest),indent=2))
print("Verified sources and data. SINGLE models,NO forest,full end-to-end gradients.",flush=True)


In [ ]:
subprocess.run([python,"-m","pytest","-q","--tb=short"],cwd=experiment,env=env,check=True)

In [ ]:
subprocess.run([python,"-u",str(experiment/"run_single.py")],cwd=experiment,env=env,check=True)

In [ ]:
subprocess.run([python,"-u",str(experiment/"analyze_single.py")],cwd=experiment,env=env,check=True)
print((out/"REPORT.md").read_text())